# 05.7 · Solutions

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/05-powerful-models/07-imbalanced-data/solutions.ipynb)

## Exercise 1

a) 19,940 / 20,000 = **99.7%**. b) Recall = 0 / 60 = **0**. c) Positive rate = 60 / 20,000 = **0.003**.
d) $w_{\text{fraud}} = 20000 / (2 \times 60) \approx 166.7$; $w_{\text{ok}} = 20000 / (2 \times 19940) \approx 0.50$.

## Exercise 2

$x_{nn} - x = (4, -1)$.

- $\lambda = 0.1$: $(10.4, 1.9)$
- $\lambda = 0.5$: $(12, 1.5)$
- $\lambda = 0.9$: $(13.6, 1.1)$

All three lie on the straight segment between the two real points; $\lambda$ says how far along.

In [1]:
# --- data helper: works locally and on Colab -------------------------------
from pathlib import Path
import urllib.request

def data_path(name):
    """Return a local path to data/<name>, downloading from GitHub on Colab."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data" / name
        if candidate.exists():
            return candidate
    url = f"https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/data/{name}"
    dest = Path(name)
    if not dest.exists():
        urllib.request.urlretrieve(url, dest)
    return dest

import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")

import sys
sys.path.insert(0, str(data_path("download.py").parent))
from download import load_retail
from sklearn.model_selection import train_test_split

raw = load_retail("Year 2010-2011")
sales = raw[~raw["Invoice"].astype(str).str.startswith("C")]
sales = sales[(sales["Quantity"] > 0) & (sales["Price"] > 0)].dropna(subset=["Customer ID"]).drop_duplicates()
sales = sales.assign(Revenue=sales["Quantity"] * sales["Price"])

cutoff = pd.Timestamp("2011-09-01")                       # features before, target after
past = sales[sales["InvoiceDate"] < cutoff]
future = sales[(sales["InvoiceDate"] >= cutoff) & (sales["InvoiceDate"] < "2011-12-01")]

cust = past.groupby("Customer ID").agg(orders=("Invoice", "nunique"), spend=("Revenue", "sum"),
                                       products=("StockCode", "nunique"),
                                       first_order=("InvoiceDate", "min"), last_order=("InvoiceDate", "max"))
cust["tenure_days"] = (cutoff - cust["first_order"]).dt.days
cust["days_since_last"] = (cutoff - cust["last_order"]).dt.days
future_spend = future.groupby("Customer ID")["Revenue"].sum().reindex(cust.index).fillna(0)
cust["churned"] = (future_spend == 0).astype(int)        # 1 = did not come back
cust = cust.drop(columns=["first_order", "last_order"])

features = ["orders", "spend", "products", "tenure_days", "days_since_last"]
X_train, X_test, y_train, y_test = train_test_split(
    cust[features], cust["churned"], test_size=0.25, random_state=42, stratify=cust["churned"])
print(f"{len(cust):,} customers, churn rate {cust['churned'].mean():.1%}, "
      f"train {len(X_train):,} / test {len(X_test):,}")

# New, rare target: the top 5% of customers by spend in Sep-Nov 2011
threshold_gbp = future_spend.quantile(0.95)
cust["big_spender"] = (future_spend > threshold_gbp).astype(int)
Xb_train, Xb_test, yb_train, yb_test = train_test_split(
    cust[features], cust["big_spender"], test_size=0.25, random_state=42, stratify=cust["big_spender"])
print(f"big spender = more than £{threshold_gbp:,.0f} next quarter")
print(f"train: {yb_train.sum()} of {len(yb_train):,} ({yb_train.mean():.1%})   "
      f"test: {yb_test.sum()} of {len(yb_test):,} ({yb_test.mean():.1%})")

from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score)

def report(name, y_true, proba, t=0.5):
    pred = (proba >= t).astype(int)
    return {"model": name, "accuracy": accuracy_score(y_true, pred),
            "precision": precision_score(y_true, pred, zero_division=0),
            "recall": recall_score(y_true, pred), "F1": f1_score(y_true, pred, zero_division=0),
            "ROC-AUC": roc_auc_score(y_true, proba), "PR-AUC": average_precision_score(y_true, proba)}

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from imblearn.pipeline import make_pipeline as imb_pipeline
from imblearn.over_sampling import RandomOverSampler, SMOTE
from imblearn.under_sampling import RandomUnderSampler

LR = lambda **kw: LogisticRegression(max_iter=1000, **kw)
models = {
    "plain":              make_pipeline(StandardScaler(), LR()),
    "class_weight":       make_pipeline(StandardScaler(), LR(class_weight="balanced")),
    "random oversample":  imb_pipeline(StandardScaler(), RandomOverSampler(random_state=42), LR()),
    "random undersample": imb_pipeline(StandardScaler(), RandomUnderSampler(random_state=42), LR()),
    "SMOTE":              imb_pipeline(StandardScaler(), SMOTE(random_state=42), LR()),
}

3,317 customers, churn rate 43.6%, train 2,487 / test 830
big spender = more than £2,393 next quarter
train: 124 of 2,487 (5.0%)   test: 42 of 830 (5.1%)


In [2]:
x, x_nn = np.array([10.0, 2.0]), np.array([14.0, 1.0])     # check exercise 2
for lam in [0.1, 0.5, 0.9]:
    print(lam, x + lam * (x_nn - x))

0.1 [10.4  1.9]
0.5 [12.   1.5]
0.9 [13.6  1.1]


## Exercise 3

In [3]:
cut2 = future_spend.quantile(0.98)
y2 = (future_spend > cut2).astype(int)
X2_train, X2_test, y2_train, y2_test = train_test_split(
    cust[features], y2, test_size=0.25, random_state=42, stratify=y2)
print(f"top 2%: £{cut2:,.0f}+, {y2_train.sum()} rare customers in training")
rows = [report(name, y2_test, m.fit(X2_train, y2_train).predict_proba(X2_test)[:, 1])
        for name, m in models.items()]
pd.DataFrame(rows).set_index("model").round(3)

top 2%: £4,415+, 50 rare customers in training


,accuracy,precision,recall,F1,ROC-AUC,PR-AUC
model,,,,,,
plain,0.984,1.000,0.235,0.381,0.876,0.763
class_weight,0.978,0.484,0.882,0.625,0.917,0.820
random oversample,0.978,0.484,0.882,0.625,0.917,0.829
random undersample,0.957,0.289,0.765,0.419,0.907,0.770
SMOTE,0.980,0.500,0.882,0.638,0.917,0.812


With only 50 rare training customers the plain model at 0.5 becomes even more timid: recall 0.24, it finds 4 of the 17 test VIPs.

The surprise is PR-AUC: it **rises**, to about 0.76, even though the random baseline fell to 0.02. The very biggest spenders are easier to recognise from past spend than the merely big ones. Rarity alone does not make a problem harder; how well the features separate the classes does.

Rebalancing lifts recall to about 0.88 and F1 from 0.38 to about 0.63. The rarer the target, the more the threshold (or weighting) matters.

## Exercise 4

In [4]:
import lightgbm as lgb
G = lambda **kw: lgb.LGBMClassifier(n_estimators=200, learning_rate=0.03, num_leaves=8,
                                    verbose=-1, n_jobs=1, random_state=42, **kw)
tree_models = {"LGBM plain": G(), "LGBM class_weight": G(class_weight="balanced"),
               "LGBM SMOTE": imb_pipeline(SMOTE(random_state=42), G())}
rows = [report(name, yb_test, m.fit(Xb_train, yb_train).predict_proba(Xb_test)[:, 1])
        for name, m in tree_models.items()]
pd.DataFrame(rows).set_index("model").round(3)

,accuracy,precision,recall,F1,ROC-AUC,PR-AUC
model,,,,,,
LGBM plain,0.958,0.630,0.405,0.493,0.928,0.557
LGBM class_weight,0.914,0.341,0.738,0.466,0.934,0.574
LGBM SMOTE,0.895,0.294,0.762,0.424,0.931,0.562


Same pattern as logistic regression: the fixes trade precision for recall at 0.5, while PR-AUC (ranking quality) moves only a little, and not always upward. For boosted trees, `class_weight` (or LightGBM's `scale_pos_weight`) is the usual first choice; SMOTE rarely beats it.